# Transform Constructors Data

1. Read bronze `constructors` table
2. Keep only the columns required for analytics (Drop `url` column)
3. Standardise column names using snake_case
4. Rename columns to make them more meaningful (name -> constructor_name)
5. Remove duplicate
6. Transform values of column `nationality` to Title Case
7. Write the transformed data to silver `constructors` table

In [0]:
%run ../00-common/01.environment-config

In [0]:
bronze_table = f'{catalog_name}.{bronze_schema}.constructors'
silver_table = f'{catalog_name}.{silver_schema}.constructors'

In [0]:
from pyspark.sql import functions as F

### Step 1 - Read bronze `constructors` table


In [0]:
const_df = spark.table(bronze_table)

In [0]:
display(const_df)

### Stpe 2 - Keep only the columns required for analytics (Drop `url` column)

In [0]:
const_selected_df = const_df.drop(
    F.col("url")
)

In [0]:
display(const_selected_df)

### Step 3 & 4 - Standardise column names 
- using snake_case
- Rename columns to make them more meaningful (name → constructor_name)

In [0]:
const_renamed_df = const_selected_df.withColumnsRenamed({
    "constructorID" : 'constructor_id',
    'name' : 'constructor_name'
})

### Step 5 - Remove duplicate

In [0]:
const_distinct_df = const_renamed_df.dropDuplicates(['constructor_id'])

### Step 6 - Transform values of column `nationality` to Title Case

In [0]:
const_final_df = (
    const_distinct_df
        .withColumn('nationality', F.initcap(F.col("nationality")))
)

### Step 7 - Write the transformed data to silver `constructors` table

In [0]:
(
    const_final_df
    .write
    .format('delta')
    .mode('overwrite')
    .saveAsTable(silver_table)
)

In [0]:
display(spark.table(silver_table))